In [3]:
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from datetime import datetime
from enum import Enum

class JobSource(Enum):
    """Enum for job sources"""
    LINKEDIN = "LinkedIn"
    GLASSDOOR = "Glassdoor"
    INDEED = "Indeed"
    MONSTER = "Monster"
    REMOTEOK = "RemoteOK"
    COMPANY_WEBSITE = "Company Website"
    RECRUITER = "Recruiter"
    OTHER = "Other"

@dataclass
class Job:
    title: str
    company: str
    location: str
    salary_range: Optional[str]
    description: str
    requirements: List[str]
    posted_date: str
    job_type: str  # full-time, part-time, contract, remote
    skills: List[str]
    source: JobSource  # NEW: Track where the job came from
    source_url: Optional[str] = None  # NEW: Direct link to the job posting
    company_rating: Optional[float] = None  # NEW: Company rating (for Glassdoor)
    posted_timestamp: Optional[datetime] = None  # NEW: For better date handling

class JobFilteringAgent:
    def __init__(self):
        self.filters = {}
        self.user_preferences = {}
        self.source_stats = {}  # NEW: Track statistics per source

    def set_user_preferences(self, preferences: Dict[str, Any]):
        """Set user preferences for job filtering"""
        self.user_preferences = {
            'preferred_locations': preferences.get('locations', []),
            'preferred_job_types': preferences.get('job_types', []),
            'min_salary': preferences.get('min_salary', None),
            'required_skills': preferences.get('required_skills', []),
            'preferred_companies': preferences.get('companies', []),
            'max_commute_time': preferences.get('max_commute_time', None),
            'remote_only': preferences.get('remote_only', False),
            'experience_level': preferences.get('experience_level', 'all'),
            'keywords_include': preferences.get('keywords_include', []),
            'keywords_exclude': preferences.get('keywords_exclude', []),
            'preferred_sources': preferences.get('preferred_sources', []),  # NEW: Filter by source
            'min_company_rating': preferences.get('min_company_rating', None)  # NEW: Min rating
        }

    def filter_jobs(self, jobs: List[Job]) -> List[Dict[str, Any]]:
        """Filter jobs based on user preferences"""
        filtered_jobs = []
        self.source_stats = {}  # Reset stats

        for job in jobs:
            score = 0
            reasons = []

            # Track source statistics
            source_name = job.source.value
            self.source_stats[source_name] = self.source_stats.get(source_name, 0) + 1

            # Source preference filter (NEW)
            if self.user_preferences['preferred_sources']:
                if job.source.value not in self.user_preferences['preferred_sources']:
                    reasons.append(f"Source {job.source.value} not preferred")
                    continue
                else:
                    score += 5  # Small bonus for preferred sources

            # Company rating filter (NEW)
            if self.user_preferences['min_company_rating'] and job.company_rating:
                if job.company_rating >= self.user_preferences['min_company_rating']:
                    score += 15
                else:
                    reasons.append(f"Company rating {job.company_rating} below minimum")
                    continue

            # Location filter
            if self.user_preferences['preferred_locations']:
                if any(loc.lower() in job.location.lower() for loc in self.user_preferences['preferred_locations']):
                    score += 20
                else:
                    reasons.append(f"Location {job.location} not preferred")
                    continue

            # Job type filter
            if self.user_preferences['preferred_job_types']:
                if job.job_type.lower() in [jt.lower() for jt in self.user_preferences['preferred_job_types']]:
                    score += 15
                else:
                    reasons.append(f"Job type {job.job_type} not preferred")
                    continue

            # Remote only filter
            if self.user_preferences['remote_only']:
                if 'remote' not in job.location.lower() and job.job_type.lower() != 'remote':
                    reasons.append("Not remote")
                    continue
                else:
                    score += 10

            # Skills match
            if self.user_preferences['required_skills']:
                matched_skills = [skill for skill in self.user_preferences['required_skills']
                                if any(skill.lower() in req.lower() for req in job.requirements)]
                if matched_skills:
                    score += len(matched_skills) * 5
                else:
                    reasons.append(f"No matching skills found")
                    continue

            # Company preference
            if self.user_preferences['preferred_companies']:
                if any(company.lower() in job.company.lower() for company in self.user_preferences['preferred_companies']):
                    score += 15

            # Salary filter
            if self.user_preferences['min_salary'] and job.salary_range:
                salary_numbers = re.findall(r'\d+', job.salary_range)
                if salary_numbers:
                    avg_salary = sum(map(int, salary_numbers)) / len(salary_numbers)
                    # Handle k/m suffixes
                    if 'k' in job.salary_range.lower():
                        avg_salary = avg_salary * 1000
                    elif 'm' in job.salary_range.lower():
                        avg_salary = avg_salary * 1000000

                    if avg_salary >= self.user_preferences['min_salary']:
                        score += 20
                    else:
                        reasons.append(f"Salary {job.salary_range} below minimum")

            # Keywords include
            if self.user_preferences['keywords_include']:
                has_keywords = any(keyword.lower() in job.description.lower() or
                                 keyword.lower() in job.title.lower()
                                 for keyword in self.user_preferences['keywords_include'])
                if not has_keywords:
                    reasons.append(f"Missing required keywords")
                    continue
                else:
                    score += 15

            # Keywords exclude
            if self.user_preferences['keywords_exclude']:
                has_excluded = any(keyword.lower() in job.description.lower() or
                                 keyword.lower() in job.title.lower()
                                 for keyword in self.user_preferences['keywords_exclude'])
                if has_excluded:
                    reasons.append(f"Contains excluded keywords")
                    continue

            # Experience level matching
            if self.user_preferences['experience_level'] != 'all':
                level_score = self._match_experience_level(job)
                if level_score == 0:
                    reasons.append(f"Experience level mismatch")
                    continue
                else:
                    score += level_score

            # Add source bonus (NEW: trust bonus for certain sources)
            if job.source in [JobSource.LINKEDIN, JobSource.GLASSDOOR]:
                score += 5  # Slight bonus for major job boards

            filtered_jobs.append({
                'job': job,
                'match_score': score,
                'reasons': reasons if reasons else ["All criteria matched"]
            })

        # Sort by match score
        filtered_jobs.sort(key=lambda x: x['match_score'], reverse=True)
        return filtered_jobs

    def _match_experience_level(self, job: Job) -> int:
        """Match job requirements with preferred experience level"""
        job_description = ' '.join(job.requirements + [job.description]).lower()

        level_keywords = {
            'entry': ['entry', 'junior', 'associate', '0-2 years', '1+ years'],
            'mid': ['mid-level', 'mid level', 'senior', '3+ years', '4+ years', '5+ years'],
            'senior': ['senior', 'lead', 'principal', '7+ years', '8+ years', '10+ years']
        }

        target_level = self.user_preferences['experience_level']
        target_keywords = level_keywords.get(target_level, [])

        for keyword in target_keywords:
            if keyword in job_description:
                return 10
        return 0

    def get_source_stats(self) -> Dict[str, int]:
        """Get statistics about job sources"""
        return self.source_stats

    def get_job_summary(self, filtered_jobs: List[Dict[str, Any]], top_n: int = 5, include_source_stats: bool = True) -> str:
        """Generate a summary of filtered jobs with source information"""
        if not filtered_jobs:
            return "No matching jobs found based on your preferences."

        summary = f"🏆 Top {min(top_n, len(filtered_jobs))} Matching Jobs:\n\n"

        for i, job_data in enumerate(filtered_jobs[:top_n], 1):
            job = job_data['job']
            summary += f"{i}. {job.title} at {job.company}\n"
            summary += f"   📍 Location: {job.location}\n"
            summary += f"   💰 Salary: {job.salary_range or 'Not specified'}\n"
            summary += f"   📋 Type: {job.job_type}\n"
            summary += f"   🔗 Source: {job.source.value}"  # NEW: Show source
            if job.source_url:
                summary += f" ({job.source_url[:50]}...)"  # Show partial URL
            summary += f"\n"
            summary += f"   🎯 Match Score: {job_data['match_score']}%\n"
            summary += f"   ✅ Match reasons: {', '.join(job_data['reasons'][:2])}\n"
            summary += f"   🛠️ Key skills needed: {', '.join(job.skills[:3])}\n"
            if job.company_rating:
                summary += f"   ⭐ Company Rating: {job.company_rating}/5.0\n"
            summary += "\n"

        # Add source statistics
        if include_source_stats and self.source_stats:
            summary += "📊 Source Statistics:\n"
            summary += "-" * 40 + "\n"
            total_jobs = sum(self.source_stats.values())
            for source, count in sorted(self.source_stats.items(), key=lambda x: x[1], reverse=True):
                percentage = (count / total_jobs) * 100
                summary += f"   • {source}: {count} jobs ({percentage:.1f}%)\n"

        return summary

    def filter_by_source(self, jobs: List[Job], sources: List[JobSource]) -> List[Job]:
        """Helper method to filter jobs by specific sources"""
        return [job for job in jobs if job.source in sources]

    def get_jobs_by_source(self, jobs: List[Job]) -> Dict[JobSource, List[Job]]:
        """Group jobs by their source"""
        jobs_by_source = {}
        for job in jobs:
            if job.source not in jobs_by_source:
                jobs_by_source[job.source] = []
            jobs_by_source[job.source].append(job)
        return jobs_by_source


# Example usage with different sources
def create_demo_jobs_with_sources() -> List[Job]:
    """Create sample jobs from different sources"""
    return [
        # LinkedIn jobs
        Job(
            title="Senior Python Developer",
            company="TechCorp",
            location="San Francisco, CA",
            salary_range="$120k - $160k",
            description="Building scalable backend services",
            requirements=["5+ years Python", "Django", "PostgreSQL", "AWS"],
            posted_date="2024-01-15",
            job_type="full-time",
            skills=["Python", "Django", "AWS"],
            source=JobSource.LINKEDIN,
            source_url="https://linkedin.com/jobs/123",
            company_rating=4.2
        ),

        # Glassdoor job with rating
        Job(
            title="Data Engineer",
            company="DataFlow",
            location="Seattle, WA",
            salary_range="$130k - $170k",
            description="Build data pipelines and ETL processes",
            requirements=["4+ years Python", "Spark", "Airflow", "AWS"],
            posted_date="2024-01-14",
            job_type="full-time",
            skills=["Python", "Spark", "Airflow"],
            source=JobSource.GLASSDOOR,
            source_url="https://glassdoor.com/jobs/456",
            company_rating=4.5
        ),

        # Indeed job
        Job(
            title="Remote Data Scientist",
            company="AI Innovations",
            location="Remote",
            salary_range="$100k - $140k",
            description="ML model development and analysis",
            requirements=["3+ years data science", "Python", "TensorFlow", "SQL"],
            posted_date="2024-01-14",
            job_type="remote",
            skills=["Python", "TensorFlow", "Machine Learning"],
            source=JobSource.INDEED,
            source_url="https://indeed.com/jobs/789",
            company_rating=3.8
        ),

        # RemoteOK job
        Job(
            title="Full Stack Developer",
            company="StartupHub",
            location="Remote",
            salary_range="$90k - $120k",
            description="Build web applications with React and Node.js",
            requirements=["3+ years React", "Node.js", "MongoDB"],
            posted_date="2024-01-13",
            job_type="remote",
            skills=["React", "Node.js", "MongoDB"],
            source=JobSource.REMOTEOK,
            source_url="https://remoteok.com/jobs/101",
            company_rating=4.0
        ),

        # Company website job
        Job(
            title="Junior Web Developer",
            company="CreativeSoft",
            location="Austin, TX",
            salary_range="$60k - $80k",
            description="Frontend development with React",
            requirements=["1+ years React", "JavaScript", "CSS"],
            posted_date="2024-01-12",
            job_type="full-time",
            skills=["React", "JavaScript", "HTML/CSS"],
            source=JobSource.COMPANY_WEBSITE,
            source_url="https://creativesoft.com/careers/202",
            company_rating=3.5
        ),

        # Monster job
        Job(
            title="DevOps Engineer",
            company="CloudTech",
            location="New York, NY",
            salary_range="$110k - $150k",
            description="Manage cloud infrastructure and CI/CD pipelines",
            requirements=["3+ years DevOps", "Docker", "Kubernetes", "Terraform"],
            posted_date="2024-01-11",
            job_type="full-time",
            skills=["Docker", "Kubernetes", "AWS"],
            source=JobSource.MONSTER,
            source_url="https://monster.com/jobs/303",
            company_rating=4.1
        )
    ]


# Advanced: Multi-source job aggregator
class MultiSourceJobAggregator(JobFilteringAgent):
    """Extended agent that can fetch from multiple sources"""

    def __init__(self):
        super().__init__()
        self.source_apis = {}  # Would store API clients for different sources

    def register_source(self, source_name: JobSource, api_client):
        """Register an API client for a job source"""
        self.source_apis[source_name] = api_client

    def fetch_from_all_sources(self, search_params: Dict) -> List[Job]:
        """Fetch jobs from all registered sources"""
        all_jobs = []

        for source, api_client in self.source_apis.items():
            try:
                print(f"🔍 Fetching jobs from {source.value}...")
                # In real implementation, this would call the respective API
                # jobs = api_client.search_jobs(search_params)
                # all_jobs.extend(jobs)
                pass
            except Exception as e:
                print(f"⚠️ Error fetching from {source.value}: {e}")

        return all_jobs

    def get_source_reliability_report(self, jobs: List[Job]) -> str:
        """Generate a report on which sources provide the best matches"""
        report = "📈 Source Reliability Report\n"
        report += "=" * 40 + "\n\n"

        # Group jobs by source and calculate average match scores
        source_scores = {}
        source_counts = {}

        # This would use historical data to calculate which sources
        # consistently provide better matches
        for job in jobs:
            source = job.source.value
            source_counts[source] = source_counts.get(source, 0) + 1

        report += "Job Distribution by Source:\n"
        for source, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
            report += f"  • {source}: {count} jobs\n"

        return report


# Initialize and run the enhanced agent
if __name__ == "__main__":
    # Create agent
    agent = JobFilteringAgent()

    # Set user preferences including source preferences
    user_prefs = {
        'locations': ['Remote', 'San Francisco', 'Seattle'],
        'job_types': ['full-time', 'remote'],
        'min_salary': 90000,
        'required_skills': ['Python', 'Machine Learning', 'Data'],
        'remote_only': True,
        'experience_level': 'mid',
        'keywords_include': ['development', 'engineering', 'data'],
        'keywords_exclude': ['junior', 'entry'],
        'preferred_sources': ['LinkedIn', 'Glassdoor', 'Indeed'],  # NEW: Prefer these sources
        'min_company_rating': 3.5  # NEW: Minimum company rating
    }

    agent.set_user_preferences(user_prefs)

    # Get jobs from multiple sources
    jobs = create_demo_jobs_with_sources()

    # Filter jobs
    matching_jobs = agent.filter_jobs(jobs)

    # Display results with source information
    print(agent.get_job_summary(matching_jobs, top_n=5, include_source_stats=True))

    # Display source statistics separately
    print("\n📊 Source Statistics:")
    print("-" * 40)
    for source, count in agent.get_source_stats().items():
        print(f"   {source}: {count} jobs")

    # Group jobs by source
    jobs_by_source = agent.get_jobs_by_source(jobs)
    print(f"\n📂 Jobs grouped by source:")
    for source, source_jobs in jobs_by_source.items():
        print(f"   {source.value}: {len(source_jobs)} jobs")
        for job in source_jobs[:2]:  # Show first 2 jobs from each source
            print(f"      - {job.title} at {job.company}")

🏆 Top 1 Matching Jobs:

1. Remote Data Scientist at AI Innovations
   📍 Location: Remote
   💰 Salary: $100k - $140k
   📋 Type: remote
   🔗 Source: Indeed (https://indeed.com/jobs/789...)
   🎯 Match Score: 120%
   ✅ Match reasons: All criteria matched
   🛠️ Key skills needed: Python, TensorFlow, Machine Learning
   ⭐ Company Rating: 3.8/5.0

📊 Source Statistics:
----------------------------------------
   • LinkedIn: 1 jobs (16.7%)
   • Glassdoor: 1 jobs (16.7%)
   • Indeed: 1 jobs (16.7%)
   • RemoteOK: 1 jobs (16.7%)
   • Company Website: 1 jobs (16.7%)
   • Monster: 1 jobs (16.7%)


📊 Source Statistics:
----------------------------------------
   LinkedIn: 1 jobs
   Glassdoor: 1 jobs
   Indeed: 1 jobs
   RemoteOK: 1 jobs
   Company Website: 1 jobs
   Monster: 1 jobs

📂 Jobs grouped by source:
   LinkedIn: 1 jobs
      - Senior Python Developer at TechCorp
   Glassdoor: 1 jobs
      - Data Engineer at DataFlow
   Indeed: 1 jobs
      - Remote Data Scientist at AI Innovations
   Remote